# Notebook to cut different event recordings
Here is the code to use when cutting recordings is necessary<br>
The recordings might contain more than one trajectory, so cutting is required

In [1]:
import sys
import h5py
import numpy as np
sys.path.append("../src/utils")
import eventIO
from src.utils.IEBCS.event_buffer import EventBuffer

Extract one trajectory from Thomas' recordings

In [ ]:
ev = eventIO.load_hdf5_metavision("/data/lkolmar/rec_thomas.hdf5")
eventIO.print_event_info(ev)

Event Buffer Information:
Number of events: 4039319
Time range [us]: 4148 - 42388067
Total time [s]: 42.383919
Recalculated RPS (using 2.0 total rounds): 0.05 RPS
Resolution: 1280 x 720
Polarity values: [0 1]
Events per pixel: 4.38
Sample events (first 10):
Event 0: t=4148, x=91, y=89, p=1
Event 1: t=4226, x=740, y=326, p=0
Event 2: t=4442, x=769, y=655, p=0
Event 3: t=4452, x=879, y=710, p=0
Event 4: t=4462, x=131, y=165, p=0
Event 5: t=4490, x=605, y=489, p=0
Event 6: t=4622, x=903, y=264, p=0
Event 7: t=4959, x=1275, y=461, p=1
Event 8: t=4969, x=122, y=294, p=0
Event 9: t=5185, x=420, y=164, p=0


In [ ]:
t_start = 9280000
t_end =   9445000

t_start_ms = int(t_start / 1000)
t_end_ms = int(t_end / 1000)
ms_to_idx = eventIO.generate_ms_to_idx(ev.get_ts())

start_idx = ms_to_idx[t_start_ms]
end_idx = ms_to_idx[t_end_ms]
print(f"Start index: {start_idx}, End index: {end_idx}, Total events: {end_idx - start_idx}")
buf = EventBuffer(0)
buf.x = ev.get_x()[start_idx:end_idx]
buf.y = ev.get_y()[start_idx:end_idx]
buf.p = ev.get_p()[start_idx:end_idx]
buf.ts = ev.get_ts()[start_idx:end_idx]
buf.i = buf.ts.shape[0]

eventIO.print_event_info(buf)

Start index: 76814, End index: 288375, Total events: 211561
Event Buffer Information:
Number of events: 211561
Time range [us]: 9280000 - 9444998
Total time [s]: 0.164998
Recalculated RPS (using 2.0 total rounds): 12.12 RPS
Resolution: 1279 x 720
Polarity values: [0 1]
Events per pixel: 0.23
Sample events (first 10):
Event 0: t=9280000, x=864, y=705, p=1
Event 1: t=9280002, x=852, y=714, p=1
Event 2: t=9280004, x=844, y=716, p=1
Event 3: t=9280005, x=850, y=700, p=1
Event 4: t=9280005, x=870, y=714, p=1
Event 5: t=9280008, x=855, y=701, p=1
Event 6: t=9280014, x=842, y=703, p=1
Event 7: t=9280015, x=858, y=704, p=1
Event 8: t=9280021, x=867, y=708, p=1
Event 9: t=9280035, x=852, y=705, p=1


In [14]:
eventIO.create_video(buf, "../data/simulator/output/max/simulation_compare_thomas.avi", resolution=(1280, 720), fps=30.0, tw=100)

/home/lkolmar/dev/Master_Thesis/notebooks/../src/utils/eventIO.py:247: RuntimeWarning: invalid value encountered in cast
  img[ind] = 125 + (2 * indsurface[ind] - 1) * np.exp(-(t + tw - tsurface[ind].astype(np.float32))/ (tw/30)) * 125


In [2]:
def cut_buffer(start_us, end_us, events):
    t_start_ms = int(start_us / 1000)
    t_end_ms = int(end_us / 1000)
    ms_to_idx = eventIO.generate_ms_to_idx(events.get_ts())

    start_idx = ms_to_idx[t_start_ms]
    end_idx = ms_to_idx[t_end_ms]
    print(f"Start index: {start_idx}, End index: {end_idx}, Total events: {end_idx - start_idx}")
    buf = EventBuffer(0)
    buf.x = events.get_x()[start_idx:end_idx]
    buf.y = events.get_y()[start_idx:end_idx]
    buf.p = events.get_p()[start_idx:end_idx]
    buf.ts = events.get_ts()[start_idx:end_idx]
    buf.i = buf.ts.shape[0]
    return buf

## Extract from Max

In [3]:
filename = "/data/lkolmar/recordings/cut/data/top/recording_2024-11-28_13-08-18_spin-4_top_speed13_cut_3.hdf5"
save_to = "../data/simulator/output/max/snippet_max_2.hdf5"

In [ ]:
ev = eventIO.load_hdf5_metavision(filename)
eventIO.print_event_info(ev)
eventIO.save_hdf5(ev, save_to, [0])

Event Buffer Information:
Number of events: 990492
Time range [us]: 10000161 - 10999998
Total time [s]: 0.999837
Recalculated RPS (using 2.0 total rounds): 2.00 RPS
Resolution: 1251 x 720
Polarity values: [0 1]
Events per pixel: 1.10
Sample events (first 10):
Event 0: t=10000161, x=307, y=49, p=0
Event 1: t=10000174, x=1188, y=29, p=0
Event 2: t=10000192, x=634, y=319, p=1
Event 3: t=10000197, x=1137, y=233, p=1
Event 4: t=10000284, x=906, y=491, p=0
Event 5: t=10000391, x=746, y=301, p=1
Event 6: t=10000406, x=1132, y=381, p=1
Event 7: t=10000520, x=1026, y=414, p=1
Event 8: t=10000788, x=401, y=88, p=0
Event 9: t=10000811, x=1193, y=585, p=1


In [5]:
ev = eventIO.load_hdf5(save_to)
# rescale ts
t_min = ev.get_ts().min()
ev.ts = ev.get_ts() - t_min
# eventIO.print_event_info(ev)
#ev = cut_buffer(830000, 955000, ev)
ev.ts = ev.get_ts() - ev.get_ts().min()
eventIO.print_event_info(ev)
eventIO.create_video(ev, "../data/simulator/output/max/snippet_max_2.avi", resolution=(1280, 720), fps=30.0, tw=280)
# eventIO.save_hdf5(ev, save_to, [0])

Event Buffer Information:
Number of events: 935706
Time range [us]: 0 - 124999
Total time [s]: 0.124999
Recalculated RPS (using 2.0 total rounds): 16.00 RPS
Resolution: 1251 x 719
Polarity values: [0 1]
Events per pixel: 1.04
Sample events (first 10):
Event 0: t=0, x=60, y=274, p=1
Event 1: t=0, x=60, y=250, p=1
Event 2: t=0, x=57, y=292, p=1
Event 3: t=1, x=62, y=258, p=1
Event 4: t=1, x=65, y=274, p=1
Event 5: t=2, x=55, y=284, p=1
Event 6: t=2, x=51, y=236, p=1
Event 7: t=2, x=50, y=296, p=1
Event 8: t=2, x=58, y=289, p=1
Event 9: t=3, x=54, y=295, p=1


/home/lkolmar/dev/Master_Thesis/notebooks/../src/utils/eventIO.py:247: RuntimeWarning: invalid value encountered in cast
  img[ind] = 125 + (2 * indsurface[ind] - 1) * np.exp(-(t + tw - tsurface[ind].astype(np.float32))/ (tw/30)) * 125
